In [1]:
import os
import pandas as pd
from bradesco_etl_silver import processar_arquivos, upload_df_as_parquet_to_gcs, query_df_from_bigquery

In [4]:
CAMINHO_PLANILHAS = "C:/Users/gabriel.goncalves_mu/Desktop/Murta All/murta_consultoria/MURTA FOR REAL/POWER BI/Atualizacao Local/Bradesco/planilhas_recebidas/"

arquivos = [
    os.path.join(CAMINHO_PLANILHAS, f)
    for f in os.listdir(CAMINHO_PLANILHAS)
    if f.lower().endswith((".xlsx", ".xls"))]

In [5]:
ofensores_list = []
internacao_list = []
for arquivo in arquivos[-1:]:
    ofensores, despesas, internacao, pacientes = processar_arquivos([arquivo])
    base = os.path.splitext(os.path.basename(arquivo))[0]

    prefixo_ofensores = "bradesco/silver/ofensores/"
    prefixo_despesas = "bradesco/silver/despesas/"
    prefixo_internacao = "bradesco/silver/internacao/"
    prefixo_paciente = "bradesco/silver/pacientes/"

    ofensores_list.append(ofensores)
    internacao_list.append(internacao)
    
    #upload_to_gcs(local_path=arquivo, bucket_name='operadoras-bi', key=f'bradesco/bronze/{base}.xlsx')
    upload_df_as_parquet_to_gcs(ofensores, arquivo, "ofensores", bucket_name='operadoras-bi', prefix_path=prefixo_ofensores)
    upload_df_as_parquet_to_gcs(despesas, arquivo, "despesas",  bucket_name='operadoras-bi', prefix_path=prefixo_despesas)
    upload_df_as_parquet_to_gcs(internacao, arquivo, "internacao", bucket_name='operadoras-bi',  prefix_path=prefixo_internacao)
    upload_df_as_parquet_to_gcs(pacientes, arquivo, "pacientes", bucket_name='operadoras-bi',  prefix_path=prefixo_paciente)

Uploaded gs://operadoras-bi/bradesco/silver/ofensores/bradesco_2025_07_ofensores.parquet
Uploaded gs://operadoras-bi/bradesco/silver/despesas/bradesco_2025_07_despesas.parquet
Uploaded gs://operadoras-bi/bradesco/silver/internacao/bradesco_2025_07_internacao.parquet
Uploaded gs://operadoras-bi/bradesco/silver/pacientes/bradesco_2025_07_pacientes.parquet


In [3]:
output_dir = "C:/Users/gabriel.goncalves_mu/Desktop/Murta All/murta_consultoria/MURTA FOR REAL/POWER BI/Atualizacao Local/Bradesco/bases_kpi/bases_kpi"

sql_refresher = '''
CALL BQ.REFRESH_MATERIALIZED_VIEW('power-bi-data-455019.bradesco_data.kpi_grouped_data_15');
CALL BQ.REFRESH_MATERIALIZED_VIEW('power-bi-data-455019.bradesco_data.kpi_grouped_data');
CALL BQ.REFRESH_MATERIALIZED_VIEW('power-bi-data-455019.bradesco_data.kpi_ofensores_custos_geral_15');
CALL BQ.REFRESH_MATERIALIZED_VIEW('power-bi-data-455019.bradesco_data.kpi_ofensores_custos_geral');
'''

tables = [
    "bradesco_pacientes",
    "kpi_grouped_data",
    "kpi_grouped_data_15",

    "kpi_ofensores_custos_geral",
    "kpi_ofensores_custos_geral_15"
]

# build a dict of table → query
queries = {
    table: f"SELECT * FROM `power-bi-data-455019.bradesco_data.{table}`"
    for table in tables}

query_df_from_bigquery(sql_refresher)

for table, sql in queries.items():
    df = query_df_from_bigquery(sql, project="power-bi-data-455019")
    df.to_excel(f"{output_dir}/{table}.xlsx", index=False)

C:\Users\gabriel.goncalves_mu\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\table.py:1965: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
